# Clase 2 — Regresión y generalización: análisis en vivo
**Machine Learning · Especialización en Analítica de Datos · Corporación Universitaria Americana**
Dr. Oscar Loyola Valenzuela

Este cuaderno acompaña la presentación de la Clase 2. Como en las láminas, cada tema se desarrolla en dos pasos: primero el **concepto**, con un ejemplo pequeño o sintético que puede seguirse a mano, y luego su **aplicación** al conjunto **California Housing** (Pace y Barry, 1997). Las figuras y tablas conservan la numeración de la presentación.

| Bloque | Concepto | Aplicación |
|---|---|---|
| 1. Generalización | Subajuste y sobreajuste (Figura 1) | Partición 80/20 (Figura 2) |
| 2. Regresión simple | Inversión publicitaria y ventas (Figura 3) | Ingreso y valor de la vivienda (Figura 4) |
| 3. Ajuste del modelo | Residuos y ECM (Figura 5); descenso de gradiente (Figura 7) | Residuos (Figura 6); gradiente (Figura 8) |
| 4. Preparación | *One-hot* (Tabla 3) y estandarización | `Pipeline` (Figura 9) |
| 5. Regresión múltiple | Precio de un departamento | Coeficientes (Figura 10) |
| 6. Métricas | Cálculo a mano (Tabla 4) | Resultados (Tabla 5, Figura 11) |
| 7. Complejidad y regularización | Términos (Tabla 6); Ridge (Figura 13); Lasso (Tabla 7) | Figuras 12, 14 y 15 |

> **Uso en clase:** ejecutar una celda a la vez. Las celdas marcadas con ❓ contienen preguntas para discutir con el curso antes de continuar.

## 0. Preparación del entorno

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, lasso_path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})
NAVY, TEAL, AMBER, PURPLE, GRAY = "#1B2A41", "#00897B", "#D97706", "#7C3AED", "#6B7280"

def rmse(y, p):
    return mean_squared_error(y, p) ** 0.5

def metricas(modelo, X, y):
    p = modelo.predict(X)
    return {"MAE": mean_absolute_error(y, p), "RMSE": rmse(y, p), "R2": r2_score(y, p)}

URL = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
df = pd.read_csv(URL)
print("Conjunto cargado:", df.shape)

---
## Datos y hallazgos del análisis exploratorio (Tablas 1 y 2)
Aplicamos el procedimiento de la Clase 1 antes de modelar.

In [ ]:
df.head()

In [ ]:
print("Faltantes por columna:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nDistritos con el valor topado en 500 000 USD:", (df["median_house_value"] >= 500000).sum())
print("\nCategorías de ocean_proximity:")
print(df["ocean_proximity"].value_counts())

In [ ]:
# Unidades: objetivo en miles de USD; ingreso en decenas de miles de USD (como en la fuente)
df["valor"] = df["median_house_value"] / 1000
df["ingreso"] = df["median_income"]

# Variables relativas al tamaño del distrito
df["habitaciones_por_hogar"] = df["total_rooms"] / df["households"]
df["dormitorios_por_habitacion"] = df["total_bedrooms"] / df["total_rooms"]
df["personas_por_hogar"] = df["population"] / df["households"]

NUM = ["longitude", "latitude", "housing_median_age", "ingreso",
       "habitaciones_por_hogar", "dormitorios_por_habitacion", "personas_por_hogar", "households"]
CAT = ["ocean_proximity"]
df[NUM + ["valor"]].describe().round(2)

❓ **Pregunta al curso:** ¿por qué construir «habitaciones por hogar» en lugar de usar el total de habitaciones?

---
## 1. Generalización
### Concepto: subajuste y sobreajuste (Figura 1)
La **generalización** es la capacidad de predecir casos que no se usaron para construir el modelo. Ajustamos tres polinomios a 18 datos sintéticos y medimos el error en datos **nuevos** generados por la misma relación.

In [ ]:
rng = np.random.default_rng(7)
f = lambda t: np.sin(2 * np.pi * t)                       # relación real
x_ent = np.sort(rng.uniform(0, 1, 18)); y_ent = f(x_ent) + rng.normal(0, 0.22, 18)
x_nue = rng.uniform(0, 1, 500);          y_nue = f(x_nue) + rng.normal(0, 0.22, 500)

g = np.linspace(0, 1, 400)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for ax, grado in zip(axes, [1, 3, 15]):
    m = make_pipeline(PolynomialFeatures(grado), LinearRegression()).fit(x_ent[:, None], y_ent)
    e_ent = rmse(y_ent, m.predict(x_ent[:, None])); e_nue = rmse(y_nue, m.predict(x_nue[:, None]))
    ax.plot(g, f(g), "--", color=GRAY, label="Relación real")
    ax.plot(g, m.predict(g[:, None]), color=TEAL, lw=2.2, label="Modelo")
    ax.scatter(x_ent, y_ent, color=NAVY, zorder=3, label="Datos")
    ax.set_ylim(-1.9, 1.9)
    ax.set_title(f"Grado {grado}\nRMSE entrenamiento = {e_ent:.2f} | datos nuevos = {e_nue:.2f}")
axes[0].legend(loc="lower left")
fig.suptitle("Figura 1. Subajuste, ajuste adecuado y sobreajuste (datos sintéticos)", y=1.04)
plt.tight_layout(); plt.show()

❓ **Pregunta al curso:** ¿qué modelo tiene el menor error de entrenamiento? ¿Es el mejor?

### Aplicación: partición de California Housing (Figura 2)

In [ ]:
X, y = df[NUM + CAT], df["valor"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Entrenamiento: {len(X_tr):,} distritos ({len(X_tr) / len(X):.0%})")
print(f"Prueba:        {len(X_te):,} distritos ({len(X_te) / len(X):.0%})")
print("\nEl conjunto de prueba se reserva hasta la evaluación final.")

---
## 2. Regresión lineal simple
### Concepto: la recta de regresión (Ecuación 1, Figura 3)
$\hat{y} = \beta_0 + \beta_1 x$, donde $\beta_0$ es el intercepto (valor predicho cuando $x = 0$) y $\beta_1$ la pendiente (cambio en $\hat{y}$ por unidad de $x$).

Ejemplo: ventas mensuales de una empresa según su inversión publicitaria, ambas en millones.

In [ ]:
inversion = np.array([1, 2, 3, 4, 5, 6, 7, 8.0])
ventas = np.array([3.1, 4.4, 4.9, 6.3, 6.8, 8.4, 8.9, 10.2])

recta = LinearRegression().fit(inversion.reshape(-1, 1), ventas)
b0, b1 = recta.intercept_, recta.coef_[0]
print(f"Recta ajustada: ventas = {b0:.2f} + {b1:.2f} · inversión")

gx = np.linspace(0, 9, 50)
plt.figure(figsize=(6.5, 4))
plt.scatter(inversion, ventas, color=NAVY, zorder=3, label="Meses observados")
plt.plot(gx, b0 + b1 * gx, color=TEAL, lw=2.2, label=f"ŷ = {b0:.2f} + {b1:.2f} x")
plt.scatter([0], [b0], color=AMBER, s=60, zorder=4, label="Intercepto β₀")
plt.xlabel("Inversión publicitaria (millones)"); plt.ylabel("Ventas (millones)")
plt.title("Figura 3. Recta ajustada (datos ilustrativos)"); plt.legend(); plt.show()

print(f"Predicción para una inversión de 9 millones: {recta.predict([[9]])[0]:.2f} millones de ventas")

**Lectura:** cada millón adicional invertido se asocia a 0,99 millones más de ventas.

### Aplicación: ingreso y valor de la vivienda (Figura 4)

In [ ]:
simple = LinearRegression().fit(X_tr[["ingreso"]], y_tr)
b0, b1 = simple.intercept_, simple.coef_[0]
print(f"ŷ = {b0:.1f} + {b1:.1f} · ingreso")
print(f"Por cada 10 000 USD adicionales de ingreso mediano, el valor predicho aumenta en {b1 * 1000:,.0f} USD")

muestra = X_tr.assign(valor=y_tr).sample(2500, random_state=1)
topado = muestra["valor"] >= 500
gx = np.linspace(0.5, 15, 50)
plt.figure(figsize=(7, 4.3))
plt.scatter(muestra.loc[~topado, "ingreso"], muestra.loc[~topado, "valor"], s=6, alpha=0.35, color=TEAL)
plt.scatter(muestra.loc[topado, "ingreso"], muestra.loc[topado, "valor"], s=8, color=AMBER, label="Valor topado en 500")
plt.plot(gx, b0 + b1 * gx, color=NAVY, lw=2.2, label=f"ŷ = {b0:.1f} + {b1:.1f} x")
plt.xlabel("Ingreso mediano (decenas de miles de USD)"); plt.ylabel("Valor mediano (miles de USD)")
plt.title("Figura 4. Recta ajustada sobre 2 500 distritos de entrenamiento"); plt.legend(); plt.show()

❓ **Pregunta al curso:** ¿qué significa el intercepto de 44,5 en este problema?

---
## 3. ¿Cómo se ajusta el modelo?
### Concepto: residuos y mínimos cuadrados (Ecuación 2, Figura 5)
El **residuo** es $y_i - \hat{y}_i$. Se elige la recta que minimiza el error cuadrático medio, $\mathrm{ECM} = \frac{1}{n}\sum (y_i - \hat{y}_i)^2$. Comparamos a mano dos rectas candidatas para cuatro puntos.

In [ ]:
x4 = np.array([1, 2, 3, 4.0])
y4 = np.array([2, 4, 5, 7.0])

for nombre, c0, c1 in [("A: ŷ = 1 + 1,0 x", 1.0, 1.0), ("B: ŷ = 0,5 + 1,6 x", 0.5, 1.6)]:
    pred = c0 + c1 * x4
    res = y4 - pred
    print(f"Recta {nombre}")
    print(pd.DataFrame({"x": x4, "y": y4, "ŷ": pred, "residuo": res, "residuo²": res ** 2}).to_string(index=False))
    print(f"ECM = {np.mean(res ** 2):.2f}\n")

mco = LinearRegression().fit(x4.reshape(-1, 1), y4)
print(f"Recta de mínimos cuadrados calculada por scikit-learn: ŷ = {mco.intercept_:.2f} + {mco.coef_[0]:.2f} x")

La recta B coincide con la de mínimos cuadrados: ninguna otra recta logra un ECM menor que 0,05.

### Aplicación: residuos en California Housing (Figura 6)

In [ ]:
ecm_tr = np.mean((y_tr - simple.predict(X_tr[["ingreso"]])) ** 2)
print(f"ECM de entrenamiento: {ecm_tr:,.1f}  ->  raíz: {np.sqrt(ecm_tr):.1f} miles de USD")

peq = X_tr.assign(valor=y_tr).query("valor < 500").sample(14, random_state=5)
pred = simple.predict(peq[["ingreso"]])
plt.figure(figsize=(7, 4.2))
for xi, yi, pi in zip(peq["ingreso"], peq["valor"], pred):
    plt.plot([xi, xi], [yi, pi], color=AMBER, lw=1.5)
plt.scatter(peq["ingreso"], peq["valor"], color=NAVY, zorder=3, label="Valor real")
gx = np.linspace(peq["ingreso"].min() - 0.3, peq["ingreso"].max() + 0.3, 20)
plt.plot(gx, simple.intercept_ + simple.coef_[0] * gx, color=TEAL, lw=2.2, label="Recta ajustada")
plt.xlabel("Ingreso mediano"); plt.ylabel("Valor mediano (miles de USD)")
plt.title("Figura 6. Residuos de 14 distritos"); plt.legend(); plt.show()

### Concepto: descenso de gradiente (Ecuación 3, Figura 7)
Cuando no hay fórmula directa, se avanza por pasos: $\beta \leftarrow \beta - \eta\, \partial J / \partial \beta$. Probamos tres tasas de aprendizaje $\eta$ sobre $J(\beta) = (\beta - 3)^2 + 1$, cuyo mínimo está en $\beta = 3$.

In [ ]:
J = lambda b: (b - 3) ** 2 + 1
dJ = lambda b: 2 * (b - 3)                 # derivada: pendiente de la curva

bb = np.linspace(-1.5, 7.5, 200)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
for ax, eta in zip(axes, [0.05, 0.3, 0.97]):
    b, trayectoria = -1.0, []
    for _ in range(8):
        trayectoria.append(b)
        b = b - eta * dJ(b)
    t = np.array(trayectoria)
    ax.plot(bb, J(bb), color=NAVY); ax.plot(t, J(t), "o-", color=AMBER)
    ax.set_title(f"η = {eta}: β final = {trayectoria[-1]:.2f}")
    ax.set_xlabel("β")
axes[0].set_ylabel("J(β)")
fig.suptitle("Figura 7. Efecto de la tasa de aprendizaje", y=1.04)
plt.tight_layout(); plt.show()

❓ **Pregunta al curso:** ¿qué ocurriría con $\eta = 1{,}1$? *Pruébelo cambiando el valor en la celda anterior.*

### Aplicación: descenso de gradiente en California Housing (Figura 8)
Implementamos el algoritmo para la pendiente $\beta_1$, con $\beta_0$ fijo en su valor óptimo.

In [ ]:
xi, yi = X_tr["ingreso"].to_numpy(), y_tr.to_numpy()
b0_opt = simple.intercept_
ecm = lambda b1: np.mean((yi - (b0_opt + b1 * xi)) ** 2)
grad = lambda b1: -2 * np.mean(xi * (yi - (b0_opt + b1 * xi)))

b1, eta = 5.0, 0.012
print(" iter   β₁       ECM")
for k in range(8):
    print(f"{k:5d} {b1:7.2f} {ecm(b1):9.1f}")
    b1 = b1 - eta * grad(b1)
print(f"\nValor de mínimos cuadrados: β₁ = {simple.coef_[0]:.2f}")

---
## 4. Preparación de las variables
### Concepto: codificación *one-hot* (Tabla 3)
Una variable categórica de $K$ categorías se reemplaza por variables indicadoras; una se omite como **categoría de referencia**.

In [ ]:
canal = pd.DataFrame({"canal": ["En línea", "Tienda", "Teléfono", "Tienda", "En línea"]})
codificador = OneHotEncoder(drop=["En línea"], sparse_output=False)
indicadoras = codificador.fit_transform(canal)
pd.concat([canal, pd.DataFrame(indicadoras, columns=codificador.get_feature_names_out(), dtype=int)], axis=1)

### Concepto: estandarización (Ecuación 4)
$z = (x - \bar{x}) / s$: número de desviaciones estándar respecto de la media.

In [ ]:
superficie = np.array([[60.0], [80.0], [100.0]])
escalador = StandardScaler().fit(superficie)
print("Media:", escalador.mean_[0], "| desviación estándar:", round(np.sqrt(escalador.var_[0]), 2))
print("Valores estandarizados:", escalador.transform(superficie).ravel().round(2))

**Nota:** `StandardScaler` usa la desviación estándar poblacional (divide por $n$); por eso obtiene $\pm 1{,}22$ y no $\pm 1$ como el cálculo de la lámina, que divide por $n - 1$. Con muchos datos la diferencia es despreciable.

---
## 5. Regresión lineal múltiple
### Concepto: «manteniendo constantes las demás» (Ecuación 5)
Ejemplo ilustrativo: precio de un departamento (millones) $= 20 + 2{,}5\,x_1 - 0{,}8\,x_2$, con $x_1$ = superficie (m²) y $x_2$ = antigüedad (años).

In [ ]:
precio = lambda sup, ant: 20 + 2.5 * sup - 0.8 * ant
print(f"Departamento de 60 m² y 10 años: {precio(60, 10):.0f} millones")
print(f"Departamento de 70 m² y 10 años: {precio(70, 10):.0f} millones  (misma antigüedad, +10 m²)")
print(f"Departamento de 60 m² y 20 años: {precio(60, 20):.0f} millones  (misma superficie, +10 años)")

### Aplicación: `Pipeline` y coeficientes en California Housing (Figuras 9 y 10)

In [ ]:
preparacion = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
    (OneHotEncoder(drop="first", handle_unknown="ignore"), CAT))

multiple = make_pipeline(preparacion, LinearRegression())
multiple.fit(X_tr, y_tr)          # mediana, media y desviación: solo con entrenamiento

nombres = [n.split("__")[1] for n in multiple[0].get_feature_names_out()]
coef = pd.Series(multiple[-1].coef_, index=nombres).drop("ocean_proximity_ISLAND").sort_values()
coef.plot.barh(color=[TEAL if v > 0 else PURPLE for v in coef], figsize=(7, 4.5))
plt.axvline(0, color=NAVY, lw=0.8)
plt.xlabel("Coeficiente (miles de USD por desviación estándar)")
plt.title("Figura 10. Coeficientes del modelo múltiple"); plt.show()
coef.round(1).to_frame("coeficiente")

❓ **Pregunta al curso:** ¿por qué se excluyó del gráfico la categoría ISLAND?

In [ ]:
print("Distritos por categoría en entrenamiento:")
print(X_tr["ocean_proximity"].value_counts())

---
## 6. Métricas de evaluación
### Concepto: cálculo a mano (Ecuaciones 6 a 8, Tabla 4)

In [ ]:
real = np.array([200, 150, 320, 250.0])
pred = np.array([210, 140, 280, 260.0])
e = real - pred
print(pd.DataFrame({"real": real, "predicho": pred, "error": e, "error²": e ** 2}).to_string(index=False))

mae_mano = np.mean(np.abs(e))
rmse_mano = np.sqrt(np.mean(e ** 2))
r2_mano = 1 - np.sum(e ** 2) / np.sum((real - real.mean()) ** 2)
print(f"\nA mano:        MAE = {mae_mano:.1f} | RMSE = {rmse_mano:.1f} | R² = {r2_mano:.3f}")
print(f"scikit-learn:  MAE = {mean_absolute_error(real, pred):.1f} | RMSE = {rmse(real, pred):.1f} | R² = {r2_score(real, pred):.3f}")

El RMSE (21,8) supera al MAE (17,5) porque la vivienda 3 concentra un error grande.

### Aplicación: resultados en California Housing (Tabla 5, Figura 11)

In [ ]:
filas = []
for nombre, modelo, cols in [("Simple (ingreso)", simple, ["ingreso"]), ("Múltiple", multiple, None)]:
    for conjunto, XX, yy in [("Entrenamiento", X_tr, y_tr), ("Prueba", X_te, y_te)]:
        XX = XX[cols] if cols else XX
        filas.append({"modelo": nombre, "conjunto": conjunto, **metricas(modelo, XX, yy)})
pd.DataFrame(filas).round(3)

In [ ]:
p_te = multiple.predict(X_te)
plt.figure(figsize=(6, 5))
plt.scatter(y_te, p_te, s=5, alpha=0.3, color=TEAL)
plt.plot([0, 520], [0, 520], "--", color=NAVY, label="Predicción perfecta")
plt.axvline(500, color=AMBER, label="Tope de 500")
plt.xlabel("Valor real (miles de USD)"); plt.ylabel("Valor predicho (miles de USD)")
plt.title("Figura 11. Valor predicho frente al real (prueba)"); plt.legend(); plt.show()
print("Predicciones negativas:", (p_te < 0).sum())

❓ **Pregunta al curso:** ¿qué tres problemas del modelo lineal revela este gráfico?

---
## 7. Complejidad y regularización
### Concepto: términos polinomiales (Ecuación 9, Tabla 6)

In [ ]:
ejemplo = pd.DataFrame({"x1": [2.0], "x2": [3.0]})
poli = PolynomialFeatures(2, include_bias=False).fit(ejemplo)
print(pd.DataFrame(poli.transform(ejemplo), columns=poli.get_feature_names_out()).to_string(index=False))

print("\nNúmero de términos generados:")
for grado in [1, 2, 3]:
    n2 = PolynomialFeatures(grado, include_bias=False).fit(np.zeros((1, 2))).n_output_features_
    n8 = PolynomialFeatures(grado, include_bias=False).fit(np.zeros((1, 8))).n_output_features_
    print(f"  grado {grado}:  p = 2 -> {n2:3d}   |   p = 8 -> {n8:3d}")

### Aplicación: términos polinomiales en California Housing (Figura 12)

In [ ]:
def preparacion_grado(grado):
    pasos = [SimpleImputer(strategy="median"), StandardScaler()]
    if grado > 1:
        pasos.append(PolynomialFeatures(grado, include_bias=False))
    return make_column_transformer((make_pipeline(*pasos), NUM),
                                   (OneHotEncoder(drop="first", handle_unknown="ignore"), CAT))

filas = []
for grado in [1, 2, 3]:
    m = make_pipeline(preparacion_grado(grado), LinearRegression()).fit(X_tr, y_tr)
    n_var = m[0].transform(X_tr[:1]).shape[1]
    filas.append({"grado": grado, "variables": n_var,
                  "RMSE entren.": metricas(m, X_tr, y_tr)["RMSE"], "RMSE prueba": metricas(m, X_te, y_te)["RMSE"],
                  "R2 prueba": metricas(m, X_te, y_te)["R2"]})
tabla_grado = pd.DataFrame(filas)
tabla_grado.plot(x="grado", y=["RMSE entren.", "RMSE prueba"], marker="o", figsize=(6, 3.8),
                 title="Figura 12. RMSE según el grado", color=[NAVY, TEAL])
plt.xticks([1, 2, 3]); plt.ylabel("RMSE (miles de USD)"); plt.show()
tabla_grado.round(3)

❓ **Pregunta al curso:** ¿cómo se interpreta un $R^2$ negativo?

### Concepto: regularización Ridge (Ecuación 10, Figura 13)
$J(\boldsymbol\beta) = \mathrm{ECM} + \lambda \sum \beta_j^2$: se penalizan los coeficientes grandes. Volvemos al polinomio de grado 15 de la Figura 1.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for ax, lam in zip(axes, [1e-12, 1e-3, 10]):
    m = make_pipeline(PolynomialFeatures(15), StandardScaler(), Ridge(alpha=lam)).fit(x_ent[:, None], y_ent)
    coef_max = np.abs(m[-1].coef_).max()
    ax.plot(g, f(g), "--", color=GRAY); ax.plot(g, m.predict(g[:, None]), color=TEAL, lw=2.2)
    ax.scatter(x_ent, y_ent, color=NAVY, zorder=3); ax.set_ylim(-1.9, 1.9)
    etiqueta = "λ ≈ 0" if lam < 1e-6 else f"λ = {lam:g}"
    ax.set_title(f"{etiqueta}\nRMSE datos nuevos = {rmse(y_nue, m.predict(x_nue[:, None])):.2f} | máx |β| = {coef_max:,.1f}")
fig.suptitle("Figura 13. Polinomio de grado 15 con tres valores de λ", y=1.06)
plt.tight_layout(); plt.show()

Al aumentar $\lambda$, el coeficiente de mayor magnitud se reduce drásticamente y la curva se suaviza. Con $\lambda$ excesivo, el modelo subajusta.

### Aplicación: Ridge en California Housing (Figura 14)

In [ ]:
lambdas = np.logspace(-2, 4, 25)
rmse_ent, rmse_pru = [], []
for lam in lambdas:
    m = make_pipeline(preparacion_grado(3), Ridge(alpha=lam)).fit(X_tr, y_tr)
    rmse_ent.append(metricas(m, X_tr, y_tr)["RMSE"]); rmse_pru.append(metricas(m, X_te, y_te)["RMSE"])
mejor = lambdas[int(np.argmin(rmse_pru))]

plt.figure(figsize=(6.5, 4))
plt.plot(lambdas, rmse_ent, color=NAVY, label="Entrenamiento")
plt.plot(lambdas, rmse_pru, color=TEAL, label="Prueba")
plt.axvline(mejor, ls="--", color=AMBER, label=f"λ = {mejor:.0f}")
plt.xscale("log"); plt.xlabel("λ"); plt.ylabel("RMSE (miles de USD)")
plt.title("Figura 14. Ridge sobre el modelo de grado 3"); plt.legend(); plt.show()

m_mejor = make_pipeline(preparacion_grado(3), Ridge(alpha=mejor)).fit(X_tr, y_tr)
print(f"Mejor λ = {mejor:.0f} -> R² de prueba = {metricas(m_mejor, X_te, y_te)['R2']:.3f}")

### Concepto: Lasso y selección de variables (Ecuación 11, Tabla 7)
$J(\boldsymbol\beta) = \mathrm{ECM} + \lambda \sum |\beta_j|$. Simulamos 8 variables, de las cuales solo $x_1$, $x_2$ y $x_3$ influyen en $y$.

In [ ]:
rng = np.random.default_rng(1)
Xs = rng.normal(size=(200, 8))
beta_real = np.array([3, -2, 1.5, 0, 0, 0, 0, 0])
ys = Xs @ beta_real + rng.normal(0, 1.5, 200)

tabla_lasso = pd.DataFrame({
    "Real": beta_real,
    "MCO": LinearRegression().fit(Xs, ys).coef_,
    "Ridge (λ = 50)": Ridge(alpha=50).fit(Xs, ys).coef_,
    "Lasso (λ = 0,3)": Lasso(alpha=0.3).fit(Xs, ys).coef_,
}, index=[f"x{j}" for j in range(1, 9)]).T
tabla_lasso.round(2) + 0.0          # + 0.0 evita mostrar «-0.00»

In [ ]:
print("Coeficientes exactamente iguales a cero:")
for metodo in ["MCO", "Ridge (λ = 50)", "Lasso (λ = 0,3)"]:
    print(f"  {metodo:16s} {(tabla_lasso.loc[metodo] == 0).sum()} de 8")

### Aplicación: trayectoria de Lasso en California Housing (Figura 15)

In [ ]:
Z = preparacion.fit_transform(X_tr)
nombres = [n.split("__")[1] for n in preparacion.get_feature_names_out()]
alphas, coefs, _ = lasso_path(Z, y_tr - y_tr.mean(), alphas=np.logspace(-1.5, 2, 120))

destacadas = {"ingreso": TEAL, "ocean_proximity_INLAND": NAVY, "housing_median_age": AMBER,
              "latitude": PURPLE, "longitude": "#B45309"}
plt.figure(figsize=(8, 4.3))
for i, n in enumerate(nombres):
    if n == "ocean_proximity_ISLAND":
        continue
    plt.plot(alphas, coefs[i], color=destacadas.get(n, "#C7CBD1"), lw=2 if n in destacadas else 1,
             label=n if n in destacadas else None)
plt.xscale("log"); plt.gca().invert_xaxis(); plt.axhline(0, color=GRAY, lw=0.8)
plt.xlabel("λ (de mayor a menor)"); plt.ylabel("Coeficiente")
plt.title("Figura 15. Trayectoria de los coeficientes de Lasso"); plt.legend(fontsize=8); plt.show()

entrada = {n: alphas[np.argmax(np.abs(coefs[i]) > 1e-8)] for i, n in enumerate(nombres) if (np.abs(coefs[i]) > 1e-8).any()}
print("Orden de ingreso al modelo (λ al que ingresa cada variable):")
print(pd.Series(entrada).sort_values(ascending=False).round(2).head(6).to_string())

❓ **Pregunta al curso:** latitud y longitud tienen coeficientes grandes en la Figura 10, pero ingresan tarde en Lasso. ¿Por qué?

---
## Cierre: ¿cómo elegir λ?
En la Figura 14, $\lambda$ se eligió observando el **conjunto de prueba**. En un proyecto real esto es una forma de **fuga de información**: el conjunto de prueba influyó en la construcción del modelo y su error deja de ser una estimación independiente. La solución —validación cruzada— se desarrolla en la Clase 5.

---
**Referencias**
- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3.ª ed.). O'Reilly.
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2.ª ed.). Springer.
- Hoerl, A. E., & Kennard, R. W. (1970). Ridge regression: Biased estimation for nonorthogonal problems. *Technometrics*, 12(1), 55–67.
- James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*. Springer.
- Pace, R. K., & Barry, R. (1997). Sparse spatial autoregressions. *Statistics & Probability Letters*, 33(3), 291–297.
- Tibshirani, R. (1996). Regression shrinkage and selection via the lasso. *Journal of the Royal Statistical Society, Series B*, 58(1), 267–288.